In [1]:
'''
14/05/26
QuickSearch_V2 returns numpy array of candidate target names.
Search done according to parameters in param.json.
Output is tagged with V2 by default.
'''
import time
import numpy as np
import glob, os, sys
import pandas as pd
%run ./asset.ipynb
%run ./width_depth.ipynb
import multiprocessing
import json, argparse


def _clean_token(x):
    if x is None:
        return None
    s = str(x).strip().lower()
    if not s:
        return None
    return s.replace(' ', '').replace('_', '').replace('-', '')


def dedup_candidates_by_v2_groups(cands, full_v2_path):
    """Deduplicate candidate names by merged V2 group IDs from full_metadata_V2.pkl."""
    if len(cands) == 0:
        return cands

    meta_v2 = pd.read_pickle(full_v2_path)
    name_cols = [c for c in ['Reduced', 'Sanitised', 'OBJECT', 'Object'] if c in meta_v2.columns]

    # token -> group id lookup
    lookup = {}
    for _, row in meta_v2.iterrows():
        gid = int(row['New Groups'])
        for c in name_cols:
            key = _clean_token(row[c])
            if key is not None and key not in lookup:
                lookup[key] = gid

    out = []
    seen_groups = set()
    unresolved = []

    for raw in cands.tolist():
        key = _clean_token(raw)
        gid = lookup.get(key)

        # Fallback through normalization regex rules loaded from grouping_V2 notebook logic
        if gid is None and key is not None and 'normalize_reduced' in globals():
            gid = lookup.get(_clean_token(normalize_reduced(key)))

        if gid is None:
            # Keep unresolved names to avoid accidental data loss
            unresolved.append(raw)
            out.append(raw)
            continue

        if gid in seen_groups:
            continue

        seen_groups.add(gid)
        out.append(raw)

    return np.array(out, dtype=object)


if __name__ == '__main__':
    startTime = time.time()

    parser = argparse.ArgumentParser(description='How to use QuickSearch_V2', epilog='ASSET by RBW')
    parser.add_argument('--p', metavar='param.json', default='param.json', help='parameter file name (default:param.json)')
    parser.add_argument('--nice', metavar='niceness', default=15, help='niceness of job (default: 15)')
    parser.add_argument('--line', metavar='line', default='K', help='define what atomic line to use (default: "K")')
    parser.add_argument('--r', metavar='res-path', default='../results/QuickSearch_V2/', help='output path (default: ../results/QuickSearch_V2/)')
    parser.add_argument('--tag', metavar='tag', default='V2', help='suffix tag for output filename (default: V2)')
    parser.add_argument('--groups-meta', default='/home/msp25gd/Downloads/res/meta/full_metadata_V2.pkl',
                        help='path to merged V2 full metadata (default: /home/msp25gd/Downloads/res/meta/full_metadata_V2.pkl)')
    parser.add_argument('--no-group-dedup', action='store_true',
                        help='disable deduplication by merged V2 groups')
    parser.add_argument('-f', '--f', dest='connection_file', default=None,
                        help='Jupyter kernel connection file')

    args = parser.parse_args()

    os.nice(int(args.nice))

    with open(args.p) as paramfile:
        param = json.load(paramfile)

    all_paths = glob.glob(param["dataset"] + '*/')

    min_spectra = int(param.get('min_spectra', 1))
    # Require a minimum number of spectra so each target has enough epochs to form a robust group reference.
    dataset_path = [p for p in all_paths if np.load(p + 'spec/sK.npy').shape[0] >= min_spectra]
    print('Stars with >= {} spectra: {}/{}'.format(min_spectra, len(dataset_path), len(all_paths)))

    Search = ASSET(parameters=param, line=args.line)

    def quicksearch_with_rv_filter(star):
        Search.ccf = False
        spec_param = Search.spec_analysis(star)
        if spec_param is None:
            return None

        new_spectra, med, med_err = spec_param

        if Search.ccf:
            rv_shift = Search.X_corr(med.copy())
            cond100 = (Search.radial_velocity > rv_shift - 50) & (Search.radial_velocity < rv_shift + 50)

        dip_detected = False
        max_peak_seen = -np.inf

        for i in range(len(new_spectra)):
            spec = new_spectra[i]

            snr = Search.snr(spec, med, Search.spectra_err[i], med_err)
            sd = np.std(snr)
            if not np.isfinite(sd) or sd == 0:
                continue

            corr_snr = snr.copy()
            if Search.ccf:
                corr_snr[cond100] = np.nan
            corr_snr = corr_snr[Search.snr_idxrange]

            sig = corr_snr / sd

            min_detect = np.nanmin(sig)
            max_detect = np.nanmax(sig)
            if np.isfinite(max_detect):
                max_peak_seen = max(max_peak_seen, float(max_detect))

            if min_detect < Search.threshold:
                width = Search.get_width(sig)
                if width >= Search.width_filter:
                    dip_detected = True

        if dip_detected:
            if not np.isfinite(max_peak_seen):
                max_peak_seen = np.nan
            return (str(Search.target_red), float(max_peak_seen))

        return None

    with multiprocessing.Pool(param["cores"]) as pool:
        out = pool.map(quicksearch_with_rv_filter, dataset_path)

    records = [item for item in out if item is not None]
    print('------------------')

    cands_raw = np.array([item[0] for item in records], dtype=object)
    peak_raw = np.array([item[1] for item in records], dtype=float)
    print('Raw candidates:', len(cands_raw))

    if args.no_group_dedup:
        cands = cands_raw
        print('Group dedup disabled')
    else:
        cands = dedup_candidates_by_v2_groups(cands_raw, args.groups_meta)
        print('After V2 group dedup:', len(cands), '(reduced by {})'.format(len(cands_raw) - len(cands)))

    # Map deduplicated names to peak values from the raw list
    peak_lookup = {}
    for name, peak in zip(cands_raw.tolist(), peak_raw.tolist()):
        if (name not in peak_lookup) or (np.isfinite(peak) and peak > peak_lookup[name]):
            peak_lookup[name] = peak

    cands_peak = np.array([peak_lookup.get(name, np.nan) for name in cands.tolist()], dtype=float)
    peak_threshold = abs(float(Search.threshold))

    # Split dip detections with significant positive peaks into requested bins
    bin_3p5_4_mask = (cands_peak >= peak_threshold) & (cands_peak < 4.0)
    bin_4_5_mask = (cands_peak >= 4.0) & (cands_peak < 5.0)
    bin_5_plus_mask = cands_peak >= 5.0

    peak_3p5_4 = cands[bin_3p5_4_mask]
    peak_4_5 = cands[bin_4_5_mask]
    peak_5_plus = cands[bin_5_plus_mask]

    print('Peak bins among candidates:')
    print('  +{:.1f} to +4 sigma: {}'.format(peak_threshold, len(peak_3p5_4)))
    print('  +4 to +5 sigma: {}'.format(len(peak_4_5)))
    print('  +5 sigma and above: {}'.format(len(peak_5_plus)))

    tag = args.tag.strip()
    if tag:
        cands_file = 'candidates_{}sig_{}cut_{}width_{}.npy'.format(Search.threshold, Search.cutoff, Search.width_filter, tag)
        peaks_file = 'candidate_max_peak_{}sig_{}cut_{}width_{}.npy'.format(Search.threshold, Search.cutoff, Search.width_filter, tag)
    else:
        cands_file = 'candidates_{}sig_{}cut_{}width.npy'.format(Search.threshold, Search.cutoff, Search.width_filter)
        peaks_file = 'candidate_max_peak_{}sig_{}cut_{}width.npy'.format(Search.threshold, Search.cutoff, Search.width_filter)

    if not os.path.exists(args.r):
        os.makedirs(args.r)
        print("new directory {} created!".format(args.r))

    np.save(args.r + cands_file, cands)
    np.save(args.r + peaks_file, cands_peak)

    peak_bin_dirs = {
        'peak_3p5_to_4': peak_3p5_4,
        'peak_4_to_5': peak_4_5,
        'peak_5_plus': peak_5_plus,
    }

    for folder, arr in peak_bin_dirs.items():
        out_dir = os.path.join(args.r, folder)
        os.makedirs(out_dir, exist_ok=True)
        np.save(os.path.join(out_dir, 'targets.npy'), arr)

    executionTime = (time.time() - startTime)
    print('Execution time in seconds: ' + str(executionTime))

Stars with >= 3 spectra: 2887/12827
[ngc280846422] not enough spectra for the search after cutting noisy spectra!


/tmp/ipykernel_6253/3932815530.py:200: RuntimeWarning: invalid value encountered in divide
  norm_s[i,:] = spec/mean


[smcwr2] not enough spectra for the search after cutting noisy spectra!
[toobulgedwarfi] not enough spectra for the search after cutting noisy spectra!
[q2059360] not enough spectra for the search after cutting noisy spectra!
[q23570048] not enough spectra for the search after cutting noisy spectra!
[novasco94] not enough spectra for the search after cutting noisy spectra!
[blg41] not enough spectra for the search after cutting noisy spectra!
[sgr105] not enough spectra for the search after cutting noisy spectra!
[tha1526] not enough spectra for the search after cutting noisy spectra!
[png007.5+04.3] not enough spectra for the search after cutting noisy spectra!
[tha1530] not enough spectra for the search after cutting noisy spectra!
[q12490159] not enough spectra for the search after cutting noisy spectra!
[lmcn19] not enough spectra for the search after cutting noisy spectra!
[wd2241325] not enough spectra for the search after cutting noisy spectra!
[sextans1104] not enough spectra f

/tmp/ipykernel_6253/3932815530.py:200: RuntimeWarning: invalid value encountered in divide
  norm_s[i,:] = spec/mean


[ltt3864] not enough spectra for the search after cutting noisy spectra!
[quintet1] not enough spectra for the search after cutting noisy spectra!
[1130+0411] not enough spectra for the search after cutting noisy spectra!
[sdssj1425+0827] not enough spectra for the search after cutting noisy spectra!
[gc8] not enough spectra for the search after cutting noisy spectra!
[kh15d] not enough spectra for the search after cutting noisy spectra!
[ex1109] not enough spectra for the search after cutting noisy spectra!
[1048147395606] not enough spectra for the search after cutting noisy spectra!
[lmc502.08.41088] not enough spectra for the search after cutting noisy spectra!
[cvso104] not enough spectra for the search after cutting noisy spectra!
[b0227369] not enough spectra for the search after cutting noisy spectra!


...

[hen2131] not enough spectra for the search after cutting noisy spectra!
[ges160012660033245] not enough spectra for the search after cutting noisy spectra!


/tmp/ipykernel_6253/3932815530.py:200: RuntimeWarning: invalid value encountered in divide
  norm_s[i,:] = spec/mean


[ngc5882] not enough spectra for the search after cutting noisy spectra!
[sdssj0957+0610] not enough spectra for the search after cutting noisy spectra!
[j1456+1609] not enough spectra for the search after cutting noisy spectra!
[j052220.98-692001.5] not enough spectra for the search after cutting noisy spectra!
[v745sco] not enough spectra for the search after cutting noisy spectra!
[imnor] not enough spectra for the search after cutting noisy spectra!
[ogletr113] not enough spectra for the search after cutting noisy spectra!
[j0415-4357] not enough spectra for the search after cutting noisy spectra!
[j0448+0950] not enough spectra for the search after cutting noisy spectra!
[j1456+0407] not enough spectra for the search after cutting noisy spectra!
[m595] not enough spectra for the search after cutting noisy spectra!
[sculptor1446] not enough spectra for the search after cutting noisy spectra!
[j052722.11-694710.1] not enough spectra for the search after cutting noisy spectra!
[j1032

/tmp/ipykernel_6253/3932815530.py:200: RuntimeWarning: invalid value encountered in divide
  norm_s[i,:] = spec/mean


[ltt9293] not enough spectra for the search after cutting noisy spectra!
[cep74] not enough spectra for the search after cutting noisy spectra!
[cvso17] not enough spectra for the search after cutting noisy spectra!
[j053253.22-695915.4] not enough spectra for the search after cutting noisy spectra!
[sdssj14390106] not enough spectra for the search after cutting noisy spectra!
[j0839+1112] not enough spectra for the search after cutting noisy spectra!
[m794] not enough spectra for the search after cutting noisy spectra!
[source6563584900242906624] not enough spectra for the search after cutting noisy spectra!
[vhs1256a] not enough spectra for the search after cutting noisy spectra!
[mrc0200+015] not enough spectra for the search after cutting noisy spectra!
[j044920.32-690900.0] not enough spectra for the search after cutting noisy spectra!
[qsob1202074] not enough spectra for the search after cutting noisy spectra!
[lehpm494] not enough spectra for the search after cutting noisy spect

In [2]:
print('QuickSearch_V2 candidates:', len(cands))
print('Output file:', cands_file)

QuickSearch_V2 candidates: 498
Output file: candidates_-3.5sig_1.5cut_2width_V2.npy


In [3]:
# Diagnose merge-aware dedup impact on current candidate list
import os

def _norm_name_for_merge(name):
    # reuse notebook normalization if available
    if 'normalize_reduced' in globals():
        return normalize_reduced(str(name).strip().lower())
    return str(name).strip().lower()

cands_list = [str(x) for x in cands.tolist()]
norm = [_norm_name_for_merge(x) for x in cands_list]

print('Raw candidates:', len(cands_list))
print('Unique by normalized name:', len(set(norm)))
print('Potential reductions:', len(cands_list) - len(set(norm)))

# show collisions (names that collapse to same normalized key)
from collections import defaultdict
bucket = defaultdict(list)
for raw, n in zip(cands_list, norm):
    bucket[n].append(raw)
collisions = {k:v for k,v in bucket.items() if len(set(v)) > 1}
print('Colliding keys:', len(collisions))
for i, (k, v) in enumerate(collisions.items()):
    if i >= 15:
        break
    print(' ', k, '=>', sorted(set(v)))

Raw candidates: 498
Unique by normalized name: 498
Potential reductions: 0
Colliding keys: 0


In [4]:
# Diagnose group-aware dedup against V2 merged metadata
import pandas as pd

full_v2_path = '/home/msp25gd/Downloads/res/meta/full_metadata_V2.pkl'
meta_v2 = pd.read_pickle(full_v2_path)

# Build lookup from any known name token -> merged group id
# Use multiple columns because folder names can come from different naming conventions.
def _clean_token(x):
    if x is None:
        return None
    s = str(x).strip().lower()
    if not s:
        return None
    s = s.replace(' ', '').replace('_', '').replace('-', '')
    return s

name_cols = [c for c in ['Reduced', 'Sanitised', 'OBJECT', 'Object'] if c in meta_v2.columns]
lookup = {}
for _, row in meta_v2.iterrows():
    gid = int(row['New Groups'])
    for c in name_cols:
        key = _clean_token(row[c])
        if key is not None and key not in lookup:
            lookup[key] = gid

resolved = 0
unresolved = 0
groups_seen = set()
for raw in cands.tolist():
    key = _clean_token(raw)
    gid = lookup.get(key)
    if gid is None and key is not None and 'normalize_reduced' in globals():
        gid = lookup.get(_clean_token(normalize_reduced(key)))
    if gid is None:
        unresolved += 1
    else:
        resolved += 1
        groups_seen.add(gid)

print('Raw candidates:', len(cands))
print('Resolved to V2 groups:', resolved)
print('Unresolved:', unresolved)
print('Unique V2 groups among resolved candidates:', len(groups_seen))
if resolved > 0:
    print('Potential reduction (resolved only):', resolved - len(groups_seen))

Raw candidates: 498
Resolved to V2 groups: 498
Unresolved: 0
Unique V2 groups among resolved candidates: 498
Potential reduction (resolved only): 0
